In [0]:
%sql
-- ============================================================
-- ETL: bronze.ventas_raw → silver.ventas
-- Patrón: MERGE (idempotente — se puede correr N veces sin duplicar)
-- ============================================================

MERGE INTO iniciacion_deportiva.silver.ventas AS target
USING (

    WITH

    -- =====================================================
    -- PASO 1: Limpieza y casteo de tipos
    -- Bronze guarda todo STRING — acá casteamos correctamente
    -- TRY_CAST devuelve NULL si el valor no es convertible
    -- en vez de romper el pipeline
    -- =====================================================
    datos_limpios AS (
        SELECT
            TRIM(id_venta)                                   AS id_venta,
            TRY_CAST(TRIM(fecha) AS TIMESTAMP)               AS fecha,
            TRIM(id_producto)                                AS id_producto,
            TRIM(titulo_producto)                            AS titulo_producto,
            TRY_CAST(TRIM(cantidad) AS INT)                  AS cantidad,
            TRIM(categoria_id)                               AS categoria_id,
            TRY_CAST(TRIM(precio_unitario) AS DOUBLE)        AS precio_unitario,
            COALESCE(TRY_CAST(TRIM(comision) AS DOUBLE), 0)  AS comision,
            TRIM(id_cliente)                                 AS id_cliente,
            TRIM(cliente_nickname)                           AS cliente_nickname,
            LOWER(TRIM(estado))                              AS estado,
            _source,
            ingested_at
        FROM iniciacion_deportiva.bronze.ventas_raw
        WHERE
            -- Filtros de calidad: descartamos filas que no tienen sentido
            id_venta IS NOT NULL AND TRIM(id_venta) != ''
            AND fecha IS NOT NULL AND TRIM(fecha) != ''
            AND id_producto IS NOT NULL AND TRIM(id_producto) != ''
            AND TRY_CAST(TRIM(precio_unitario) AS DOUBLE) > 0
            AND TRY_CAST(TRIM(cantidad) AS INT) > 0
            AND estado IS NOT NULL AND TRIM(estado) != ''
    ),

    -- =====================================================
    -- PASO 2: Deduplicación
    -- Bronze tiene duplicados intencionales (ventana 2hs,
    -- run cada 15 min). Nos quedamos con el registro más
    -- reciente por id_venta — si el estado cambió
    -- (ej: paid → partially_refunded) el MERGE lo actualiza
    -- =====================================================
    datos_deduplicados AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY id_venta
                ORDER BY ingested_at DESC
            ) AS rn
        FROM datos_limpios
    ),

    -- =====================================================
    -- PASO 3: Ensamblar el registro final
    -- título tal como vino en la venta — Gold maneja el SCD
    -- =====================================================
    datos_finales AS (
        SELECT
            id_venta,
            fecha,
            id_producto,
            titulo_producto,              -- sin normalizar, Gold maneja SCD Type 2
            cantidad,
            categoria_id,
            precio_unitario,
            comision,
            id_cliente,
            cliente_nickname,
            estado,
            _source,
            ingested_at
        FROM datos_deduplicados
        WHERE rn = 1                      -- un solo registro por id_venta
    )

    SELECT * FROM datos_finales

) AS source
ON target.id_venta = source.id_venta

-- Si la venta ya existe y el estado cambió → actualizamos
WHEN MATCHED AND target.estado != source.estado
    THEN UPDATE SET
        target.estado                  = source.estado,
        target._processing_timestamp   = CURRENT_TIMESTAMP()

-- Si la venta no existe → insertamos
WHEN NOT MATCHED
    THEN INSERT (
        id_venta,
        fecha,
        id_producto,
        titulo_producto,
        cantidad,
        categoria_id,
        precio_unitario,
        comision,
        id_cliente,
        cliente_nickname,
        estado,
        _source,
        _processing_timestamp
    )
    VALUES (
        source.id_venta,
        source.fecha,
        source.id_producto,
        source.titulo_producto,
        source.cantidad,
        source.categoria_id,
        source.precio_unitario,
        source.comision,
        source.id_cliente,
        source.cliente_nickname,
        source.estado,
        'iniciacion_deportiva.bronze.ventas_raw',
        CURRENT_TIMESTAMP()
    );